# 08 · Deep Network Optimization and Loss Landscapes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/08-optimization-landscapes/08_optimization_landscapes.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-08-optimization-landscapes.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 08](https://github.com/anuxlab/deep-learning-systems/wiki/Module-08) |
| **Level** | Intermediate–Advanced |
| **Typical CPU runtime** | ~1–2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 02, 06, 07 |

## What you will do

- Implement SGD, momentum, Nesterov, AdaGrad, RMSProp, Adam and AdamW from scratch.
- Verify the from-scratch Adam against `torch.optim.Adam` step by step.
- Visualise optimiser trajectories on ill-conditioned, non-convex and saddle landscapes.
- Plot learning-rate schedules (step, cosine, warm-up) and demonstrate gradient clipping.
- Run a controlled optimiser comparison on a small network.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · The optimiser zoo (update rules)
With gradient $g_t$ and learning rate $\eta$:

- **SGD**: $\theta\leftarrow\theta-\eta g$
- **Momentum**: $v\leftarrow\mu v+g,\ \theta\leftarrow\theta-\eta v$
- **Nesterov**: evaluate the gradient at the look-ahead point $\theta-\eta\mu v$
- **AdaGrad**: $G\mathrel{+}=g^2,\ \theta\leftarrow\theta-\eta g/(\sqrt G+\epsilon)$
- **RMSProp**: $G\leftarrow\rho G+(1-\rho)g^2$ (same step)
- **Adam**: $m\leftarrow\beta_1m+(1-\beta_1)g,\ v\leftarrow\beta_2v+(1-\beta_2)g^2$, bias-correct $\hat m=m/(1-\beta_1^t),\hat v=v/(1-\beta_2^t)$, $\theta\leftarrow\theta-\eta\hat m/(\sqrt{\hat v}+\epsilon)$
- **AdamW**: Adam with *decoupled* weight decay $\theta\leftarrow\theta-\eta\lambda\theta$ (not mixed into $g$).

All are implemented below as small classes sharing a `step(params, grads)` interface on NumPy arrays.

In [ ]:
class SGD:
    def __init__(s, lr=.01, momentum=0., nesterov=False): s.lr, s.mu, s.nest, s.v = lr, momentum, nesterov, None
    def step(s, p, g):
        if s.v is None: s.v = np.zeros_like(p)
        s.v = s.mu * s.v + g
        return p - s.lr * (g + s.mu * s.v if s.nest else s.v)
class AdaGrad:
    def __init__(s, lr=.1, eps=1e-8): s.lr, s.eps, s.G = lr, eps, None
    def step(s, p, g):
        s.G = g ** 2 if s.G is None else s.G + g ** 2
        return p - s.lr * g / (np.sqrt(s.G) + s.eps)
class RMSProp:
    def __init__(s, lr=.01, rho=.9, eps=1e-8): s.lr, s.rho, s.eps, s.G = lr, rho, eps, None
    def step(s, p, g):
        s.G = (1 - s.rho) * g ** 2 if s.G is None else s.rho * s.G + (1 - s.rho) * g ** 2
        return p - s.lr * g / (np.sqrt(s.G) + s.eps)
class Adam:
    def __init__(s, lr=1e-3, b1=.9, b2=.999, eps=1e-8, wd=0., decoupled=False):
        s.lr, s.b1, s.b2, s.eps, s.wd, s.dec, s.m, s.v, s.t = lr, b1, b2, eps, wd, decoupled, None, None, 0
    def step(s, p, g):
        if s.m is None: s.m, s.v = np.zeros_like(p), np.zeros_like(p)
        s.t += 1
        if s.wd and not s.dec: g = g + s.wd * p                  # classic L2 inside the gradient
        s.m = s.b1 * s.m + (1 - s.b1) * g; s.v = s.b2 * s.v + (1 - s.b2) * g ** 2
        mh, vh = s.m / (1 - s.b1 ** s.t), s.v / (1 - s.b2 ** s.t)
        if s.wd and s.dec: p = p - s.lr * s.wd * p                 # AdamW: decoupled decay (applied to the old weights, as in torch)
        return p - s.lr * mh / (np.sqrt(vh) + s.eps)
print("optimisers ready")

### Verification: our Adam/SGD/RMSProp/AdamW vs `torch.optim`
Same initial parameters, same gradient sequence → the parameter trajectories must coincide.

In [ ]:
r = np.random.default_rng(0); p0 = r.normal(size=5); grads = [r.normal(size=5) for _ in range(20)]
def run_torch(make):
    p = torch.tensor(p0.copy(), requires_grad=True); o = make([p])
    for g in grads: p.grad = torch.tensor(g); o.step()
    return p.detach().numpy()
def run_mine(o):
    p = p0.copy()
    for g in grads: p = o.step(p, g)
    return p
pairs = {
 "SGD+momentum": (SGD(.1, .9), lambda ps: torch.optim.SGD(ps, lr=.1, momentum=.9)),
 "SGD+Nesterov": (SGD(.1, .9, True), lambda ps: torch.optim.SGD(ps, lr=.1, momentum=.9, nesterov=True)),
 "RMSProp": (RMSProp(.01, .99), lambda ps: torch.optim.RMSprop(ps, lr=.01, alpha=.99, eps=1e-8)),
 "AdaGrad": (AdaGrad(.1), lambda ps: torch.optim.Adagrad(ps, lr=.1, eps=1e-8)),
 "Adam": (Adam(.01), lambda ps: torch.optim.Adam(ps, lr=.01)),
 "AdamW": (Adam(.01, wd=.1, decoupled=True), lambda ps: torch.optim.AdamW(ps, lr=.01, weight_decay=.1)),
}
for n, (mine, mk) in pairs.items():
    d = np.abs(run_mine(mine) - run_torch(mk)).max(); check(d < 1e-6, f"{n:13s} matches torch.optim (max diff {d:.1e})")

## 2 · Landscapes and trajectories
We run each optimiser on two classic 2-D problems and draw the paths.

1. **Ill-conditioned quadratic** $f=\tfrac12(x^2+50y^2)$ — $\kappa=50$; plain GD zig-zags across the narrow valley.
2. **Rosenbrock** $f=(1-x)^2+100(y-x^2)^2$ — a curved non-convex valley; minimum at $(1,1)$.

In [ ]:
def quad(p): return .5 * (p[0] ** 2 + 50 * p[1] ** 2)
def gquad(p): return np.array([p[0], 50 * p[1]])
def rosen(p): return (1 - p[0]) ** 2 + 100 * (p[1] - p[0] ** 2) ** 2
def grosen(p): return np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0] ** 2), 200 * (p[1] - p[0] ** 2)])
def run(opt, f, g, p0, steps):
    p = np.array(p0, float); path = [p.copy()]
    for _ in range(steps): p = opt.step(p, g(p)); path.append(p.copy())
    return np.array(path)
cfgs = {"quad": dict(f=quad, g=gquad, p0=[-4, 1.5], steps=80, opts=lambda: {"SGD": SGD(.035), "Momentum": SGD(.02, .9), "Nesterov": SGD(.02, .9, True), "AdaGrad": AdaGrad(1.0), "RMSProp": RMSProp(.15), "Adam": Adam(.4)}),
        "rosen": dict(f=rosen, g=grosen, p0=[-1.2, 1.0], steps=4000, opts=lambda: {"SGD": SGD(1e-3), "Momentum": SGD(2e-4, .9), "Nesterov": SGD(2e-4, .9, True), "AdaGrad": AdaGrad(.5), "RMSProp": RMSProp(.005), "Adam": Adam(.02)})}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5)); final = {}
for ax, (name, c) in zip(axes, cfgs.items()):
    if name == "quad": gx, gy = np.meshgrid(np.linspace(-5, 3, 300), np.linspace(-2, 2, 300))
    else: gx, gy = np.meshgrid(np.linspace(-1.6, 1.6, 300), np.linspace(-.5, 2, 300))
    Z = np.array([[c["f"]([a, b]) for a in gx[0]] for b in gy[:, 0]]); ax.contour(gx, gy, np.log1p(Z), 30, cmap="gray", alpha=.5)
    for oname, o in c["opts"]().items():
        path = run(o, c["f"], c["g"], c["p0"], c["steps"]); final[(name, oname)] = c["f"](path[-1]); ax.plot(*path.T, label=f"{oname} (f={final[(name, oname)]:.2g})", lw=1.2)
    ax.plot(*c["p0"], "ko"); ax.set_title({"quad": "Ill-conditioned quadratic (κ=50)", "rosen": "Rosenbrock valley"}[name]); ax.legend(fontsize=7)
show("optimizer_trajectories")
for k, v in final.items(): print(k, f"{v:.3e}")
check(final[("quad", "Momentum")] < final[("quad", "SGD")], "momentum beats plain GD on the ill-conditioned quadratic")
check(final[("rosen", "Adam")] < 1e-2, "Adam makes real progress along the Rosenbrock valley")
METRICS["landscape_final_f"] = {f"{a}/{b}": v for (a, b), v in final.items()}

## 3 · Saddle points
At a saddle (e.g. $f=x^2-y^2$ at the origin) the gradient is zero but it is not a minimum. Starting *exactly* on the unstable axis, GD stays stuck; a tiny perturbation (as SGD noise provides) lets it escape exponentially fast along the negative-curvature direction.

In [ ]:
fs = lambda p: p[0] ** 2 - p[1] ** 2 + .25 * p[1] ** 4; gs = lambda p: np.array([2 * p[0], -2 * p[1] + p[1] ** 3])
stuck = run(SGD(.1), fs, gs, [1.0, 0.0], 100); escape = run(SGD(.1), fs, gs, [1.0, 1e-6], 100)
print("start on axis  -> final", stuck[-1].round(4), " f =", round(fs(stuck[-1]), 4))
print("tiny noise     -> final", escape[-1].round(4), " f =", round(fs(escape[-1]), 4))
check(abs(stuck[-1][1]) < 1e-9 and fs(escape[-1]) < -0.9, "exact saddle traps GD (f≈0); a 1e-6 perturbation escapes to the true minima (f≈-1)")
H = np.array([[2, 0], [0, -2.]]); print("Hessian eigenvalues at the saddle:", np.linalg.eigvalsh(H))

## 4 · Learning-rate schedules and warm-up
Cosine decay: $\eta_t=\eta_{min}+\tfrac12(\eta_{max}-\eta_{min})(1+\cos\frac{\pi t}{T})$. Warm-up ramps linearly from 0 for the first $T_w$ steps (important for Adam/Transformers because early second-moment estimates are unreliable). We compare against `torch.optim.lr_scheduler`.

In [ ]:
T, Tw, lr0 = 1000, 100, 1.0; t = np.arange(T)
step = lr0 * .1 ** (t // 300); cos = .5 * lr0 * (1 + np.cos(np.pi * t / T))
warm_cos = np.where(t < Tw, lr0 * (t + 1) / Tw, .5 * lr0 * (1 + np.cos(np.pi * (t - Tw) / (T - Tw))))
lin_decay = lr0 * (1 - t / T)
for n, s in [("step", step), ("cosine", cos), ("warmup+cosine", warm_cos), ("linear", lin_decay)]: plt.plot(t, s, label=n)
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.title("LR schedules"); show("lr_schedules")
p = torch.zeros(1, requires_grad=True); o = torch.optim.SGD([p], lr=lr0); sch = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=T); vals = []
for _ in range(T): vals.append(o.param_groups[0]["lr"]); o.step(); sch.step()
check(np.allclose(vals, cos, atol=1e-9), "our cosine schedule == torch CosineAnnealingLR")

## 5 · Gradient clipping
Clipping the global norm to $c$ (rescale $g\leftarrow g\cdot c/\|g\|$ when $\|g\|>c$) prevents a single exploding gradient from destroying training — vital for RNNs/Transformers. Below, a loss surface with a **cliff**: without clipping one step jumps far away.

In [ ]:
fc = lambda p: np.where(p[0] > 0, 100 * p[0], 0.5 * p[0] ** 2) + 0 * p[1]
gc = lambda p: np.array([100.0 if p[0] > 0 else p[0], 0.0])
def clip(g, c): n = np.linalg.norm(g); return g if n <= c else g * c / n
p = np.array([1e-3, 0.]); a = p - .5 * gc(p); b = p - .5 * clip(gc(p), 1.0)
print("unclipped step ->", a, "  clipped step ->", b)
check(abs(a[0]) > 10 and abs(b[0]) < 1, "clipping tames the cliff gradient")
# torch utility computes the same thing
w = torch.zeros(3, requires_grad=True); w.grad = torch.tensor([30., 40., 0.]); tot = torch.nn.utils.clip_grad_norm_([w], 5.0)
check(abs(tot.item() - 50) < 1e-5 and abs(w.grad.norm().item() - 5) < 1e-5, "torch clip_grad_norm_: norm 50 -> 5")

## 6 · A controlled comparison on a small network
Same data, same architecture, same initial weights, same number of epochs; only the optimiser changes. We tune the learning rate on a small grid per optimiser (fairness!) using a validation split, then report test accuracy. Dataset: scikit-learn digits (8×8 images, 10 classes).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
D = load_digits(); Xd = (D.data / 16.).astype(np.float32); yd = D.target
Xa, Xte, ya, yte = train_test_split(Xd, yd, test_size=.25, random_state=0, stratify=yd)
Xtr, Xva, ytr, yva = train_test_split(Xa, ya, test_size=.2, random_state=0, stratify=ya)
T_ = lambda a: torch.tensor(a)
def train_opt(name, lr, epochs=25, seed=0):
    torch.manual_seed(seed); m = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))
    ctor = {"SGD": lambda p: torch.optim.SGD(p, lr=lr), "Momentum": lambda p: torch.optim.SGD(p, lr=lr, momentum=.9), "Nesterov": lambda p: torch.optim.SGD(p, lr=lr, momentum=.9, nesterov=True),
            "RMSProp": lambda p: torch.optim.RMSprop(p, lr=lr), "Adam": lambda p: torch.optim.Adam(p, lr=lr), "AdamW": lambda p: torch.optim.AdamW(p, lr=lr)}[name](m.parameters())
    X, y = T_(Xtr), T_(ytr); curve = []
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        for s in range(0, len(X), 32):
            b = perm[s:s + 32]; ctor.zero_grad(); l = F.cross_entropy(m(X[b]), y[b]); l.backward(); ctor.step()
        with torch.no_grad(): curve.append(F.cross_entropy(m(X), y).item())
    with torch.no_grad(): return m, curve, (m(T_(Xva)).argmax(1) == T_(yva)).float().mean().item()
grids = {"SGD": [.01, .03, .1, .3], "Momentum": [.003, .01, .03], "Nesterov": [.003, .01, .03], "RMSProp": [1e-4, 3e-4, 1e-3], "Adam": [3e-4, 1e-3, 3e-3], "AdamW": [3e-4, 1e-3, 3e-3]}
results = {}
with timer("optimizer_comparison"):
    for name, g in grids.items():
        best = max(((lr, *train_opt(name, lr)) for lr in g), key=lambda t: t[3]); lr, m, curve, va = best
        with torch.no_grad(): te = (m(T_(Xte)).argmax(1) == T_(yte)).float().mean().item()
        results[name] = dict(lr=lr, val=va, test=te, curve=curve); print(f"{name:9s} best lr={lr:<7g} val acc={va:.3f}  test acc={te:.3f}")
for k, v in results.items(): plt.semilogy(v["curve"], label=f"{k} (lr={v['lr']})")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.legend(); plt.title("Optimiser comparison, digits MLP (best LR each)"); show("optimizer_comparison")
check(all(v["test"] > .9 for v in results.values()), "every tuned optimiser exceeds 90% test accuracy")
METRICS["optimizer_test_acc"] = {k: v["test"] for k, v in results.items()}

In [ ]:
save_metrics("08", METRICS)
log.info('notebook 08 finished OK')

## Wrap-up

## Exercises (do these next)

1. Add a 5-seed repeat to section 6 and report mean ± std; are the differences between optimisers significant?
2. Implement Lion or Adafactor-style updates and verify against a reference.
3. Sweep AdamW weight decay on the digits task and plot validation accuracy.

## Where to read more

- Wiki chapter: [Module 08](https://github.com/anuxlab/deep-learning-systems/wiki/Module-08) (derivations, worked example, Q&A).
- Kingma & Ba (2015) Adam; Loshchilov & Hutter (2019) AdamW; Ruder (2016) *An overview of gradient descent optimization algorithms*.